# Lab Assignment 8 — NLP using Python-II
## UCS420 Cognitive Computing

**Name:** Jasmandeep Sharma  
**Roll Number:** 1024170269

## Q1. Text preprocessing and word frequency

In [ ]:
import re
from collections import Counter
import nltk
import pandas as pd
import matplotlib.pyplot as plt

for resource in ['punkt', 'punkt_tab', 'stopwords']:
    nltk.download(resource, quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
STOP_WORDS = set(stopwords.words('english'))

paragraph = ("Artificial intelligence is changing the way people learn and work. "
             "It helps computers recognize patterns and solve complex problems. "
             "Students can use intelligent tools to understand difficult concepts. "
             "However, technology should support human thinking rather than replace it. "
             "Responsible use of technology can save time and improve productivity. "
             "I enjoy learning about AI because it connects creativity with science.")

lower_text = paragraph.lower()
clean_text = re.sub(r'[^a-z0-9\s]', '', lower_text)
print("Original paragraph:\n", paragraph)
print("\nLowercase, punctuation removed:\n", clean_text)
print("\nSentences:", sent_tokenize(paragraph))
tokens = word_tokenize(clean_text)
print("\nWord tokens:", tokens)
print("\nPython split():", paragraph.split())
print("\nNLTK word_tokenize():", word_tokenize(paragraph))

tokens_no_stopwords = [w for w in tokens if w not in STOP_WORDS]
print("\nTokens without stopwords:", tokens_no_stopwords)
print("\nWord frequency (excluding stopwords):")
for word, count in Counter(tokens_no_stopwords).most_common():
    print(f"{word}: {count}")

## Q2. Regex extraction, stopword removal, stemming and lemmatization

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
for resource in ['wordnet', 'omw-1.4']:
    nltk.download(resource, quiet=True)

alphabetic_words = re.findall(r'\b[A-Za-z]+\b', paragraph.lower())
filtered_words = [w for w in alphabetic_words if w not in STOP_WORDS]
porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stemmed_words = [porter.stem(w) for w in filtered_words]
lemmatized_words = [lemmatizer.lemmatize(w) for w in filtered_words]

print("Alphabetic words:", alphabetic_words)
print("\nAfter stopword removal:", filtered_words)
print("\nPorter stemming:", stemmed_words)
print("\nWordNet lemmatization:", lemmatized_words)
print("\nWords where outputs differ:")
for original, stem, lemma in zip(filtered_words, stemmed_words, lemmatized_words):
    if stem != lemma:
        print(f"{original:15} -> stem: {stem:12} | lemma: {lemma}")

**Explanation:** Stemming mechanically removes or changes word endings and can produce fragments that are not dictionary words. Lemmatization aims to return a valid base form. Stemming is useful for fast retrieval when exact forms matter less; lemmatization is preferable when readability and linguistic meaning are important.

## Q3. Bag of Words and TF-IDF for three short texts

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

texts = [
    'The camera takes sharp photos and has excellent battery life.',
    'This phone has a bright screen, fast processor, and long battery life.',
    'The laptop has a fast processor and a comfortable keyboard.'
]
count_vectorizer = CountVectorizer(stop_words='english')
bow = count_vectorizer.fit_transform(texts)
display(pd.DataFrame(bow.toarray(), columns=count_vectorizer.get_feature_names_out(),
                     index=['Text 1', 'Text 2', 'Text 3']))

tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf = tfidf_vectorizer.fit_transform(texts)
terms = tfidf_vectorizer.get_feature_names_out()
display(pd.DataFrame(tfidf.toarray(), columns=terms,
                     index=['Text 1', 'Text 2', 'Text 3']).round(3))
for i, row in enumerate(tfidf.toarray()):
    top = row.argsort()[::-1][:3]
    print(f"Top 3 keywords for Text {i+1}:", [(terms[j], round(row[j], 3)) for j in top if row[j] > 0])

## Q4. Jaccard and cosine similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

ai_text = ("Artificial intelligence learns patterns from data. It can automate tasks and support decisions. "
           "AI is used in healthcare, education, and business. It improves speed but needs careful oversight.")
blockchain_text = ("Blockchain stores records in linked blocks. A distributed network verifies transactions. "
                   "It can support digital assets and transparent supply chains. Security depends on good design.")

def preprocess(text):
    return re.findall(r'\b[a-z]+\b', text.lower())

set_ai, set_blockchain = set(preprocess(ai_text)), set(preprocess(blockchain_text))
jaccard = len(set_ai & set_blockchain) / len(set_ai | set_blockchain)
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform([ai_text, blockchain_text])
cosine = cosine_similarity(vectors[0:1], vectors[1:2])[0, 0]
print("Jaccard similarity:", round(jaccard, 4))
print("Cosine similarity:", round(float(cosine), 4))

**Analysis:** Jaccard compares overlap between unique word sets and ignores repetitions. TF-IDF cosine similarity accounts for term weights and can be more informative when terms differ in importance. Interpret both in the context of these short descriptions.

## Q5. Product reviews, sentiment analysis and word cloud

In [ ]:
reviews = [
    'The headphones sound excellent and the battery lasts all day.',
    'The delivery was late and the product stopped working quickly.',
    'The design is fine, but the price is a little high.',
    'I love the clear sound and comfortable fit.',
    'The build quality is poor and customer support was unhelpful.'
]
from textblob import TextBlob
sentiment_rows, positive_reviews = [], []
for review in reviews:
    blob = TextBlob(review)
    polarity, subjectivity = blob.sentiment.polarity, blob.sentiment.subjectivity
    label = 'Positive' if polarity > 0 else ('Negative' if polarity < 0 else 'Neutral')
    sentiment_rows.append({'Review': review, 'Polarity': polarity,
                           'Subjectivity': subjectivity, 'Sentiment': label})
    if label == 'Positive':
        positive_reviews.append(review)
display(pd.DataFrame(sentiment_rows).round(3))

from wordcloud import WordCloud
if positive_reviews:
    wc = WordCloud(width=800, height=400, background_color='white').generate(' '.join(positive_reviews))
    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud — Positive Reviews')
    plt.show()

## Q6. Text generation with a simple LSTM model

In [ ]:
training_text = (
    'Technology helps people solve problems and share ideas. Artificial intelligence can learn patterns from data. '
    'A neural network studies examples and adjusts its internal weights. Language models learn which words often '
    'appear together. Training requires text, tokens, and many examples. A small model cannot understand the world, '
    'but it can learn simple patterns from a short paragraph. When a seed word is provided, the model predicts the '
    'next word from the words that came before it. Repeating this process creates a short sequence of new text. '
    'More varied training data usually produces more varied output.'
)

import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical

tokenizer = Tokenizer()
tokenizer.fit_on_texts([training_text])
sequences = []
for sentence in training_text.split('. '):
    seq = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(2, len(seq) + 1):
        sequences.append(seq[:i])

max_len = max(len(seq) for seq in sequences)
padded = pad_sequences(sequences, maxlen=max_len, padding='pre')
X_train = padded[:, :-1]
y_train = to_categorical(padded[:, -1], num_classes=len(tokenizer.word_index) + 1)

model = Sequential([
    Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=16),
    LSTM(32),
    Dense(len(tokenizer.word_index) + 1, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=100, verbose=0)
print("Model trained.")

def generate_text(seed, next_words=8):
    result = seed
    for _ in range(next_words):
        seq = tokenizer.texts_to_sequences([result])[0]
        seq = pad_sequences([seq], maxlen=max_len - 1, padding='pre')
        prediction = model.predict(seq, verbose=0)[0]
        next_id = int(np.argmax(prediction))
        next_word = next((word for word, idx in tokenizer.word_index.items() if idx == next_id), '')
        if not next_word:
            break
        result += ' ' + next_word
    return result

print("Generated line 1:", generate_text('technology', 8))
print("Generated line 2:", generate_text('artificial intelligence', 8))
print("Generated line 3:", generate_text('language models', 8))